## 4. Evaluace a interpretace výsledků

### 4.1 Srovnání klasifikačních modelů a výběr vítězného algoritmu
V rámci vývoje systému jsme testovali Logistickou regresi, Random Forest a Gradient Boosting. Pro vyvážení dat jsme využili techniku SMOTE.

| Model | Accuracy | ROC-AUC | Recall (Dropout) | F1-Score (Dropout) |
| :--- | :---: | :---: | :---: | :---: |
| **Logistic Regression** | 0,75 | 0,798 | **0,70** | **0,57** |
| **Random Forest** | 0,78 | 0,797 | 0,59 | 0,56 |
| **Gradient Boosting** | **0,79** | **0,801** | 0,49 | 0,53 |

**Vítězný model:** Logistická regrese. Nabízí nejvyšší Recall (0,70), což je klíčové pro zachycení co největšího počtu ohrožených studentů.


### 4.2 Analýza matice záměn a ekonomický dopad
Na základě matice záměn jsme aplikovali náklady: 50 000 Kč (ztráta studenta) a 5 000 Kč (náklady na pomoc).

| Scénář | Počet studentů | Finanční dopad |
| :--- | :---: | :--- |
| **Správně identifikovaný dropout (TP)** | 15 | **+ 675 000 Kč** |
| **Falešný poplach (FP)** | 7 | **- 35 000 Kč** |
| **Čistý přínos modelu** | | **+ 640 000 Kč** |

**Doporučení:** Vzhledem k vysokému počtu nezachycených odchodů (FN) doporučujeme snížit klasifikační práh (threshold), aby se maximalizoval finanční zisk skrze vyšší záchyt studentů.

### 4.3 Důležitost příznaků a shlukování
Nejsilnějším prediktorem dropoutu je **Stress_Index (0,193)** a **Attendance_Rate (0,097)**.

Pomocí loketní metody a Silhouette score (0,243) jsme identifikovali **3 optimální shluky**:
1. **Shluk 0 (Vysoké riziko):** Vysoký stres, nízké CGPA. Hlavní cíl intervence.
2. **Shluk 1 (Pracující):** Stabilní výsledky, vyšší příjmy.
3. **Shluk 2 (Vzorní):** Vysoká disciplína, minimální riziko.

### 4.4 Závěr a finální verdikt
Implementace systému je pro školu **přínosná**. Model generuje zisk **640 000 Kč** a identifikuje nejohroženější segmenty studentů. Projekt je připraven k nasazení do pilotního provozu.

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.metrics import (
    confusion_matrix, recall_score, roc_auc_score,
    classification_report
)
warnings.filterwarnings('ignore')

# Data
X_train_prep, X_test_prep, y_train, y_test = joblib.load('../data/processed/split_data.pkl')

# Tuned modely (výstup z 03d)
def load_estimator(path):
    pipeline = joblib.load(path)
    if hasattr(pipeline, 'named_steps') and 'model' in pipeline.named_steps:
        return pipeline.named_steps['model']
    return pipeline

tuned_models = {
    'Logistic Regression': load_estimator('../models/tuned/logistic_regression_tuned.pkl'),
    'Random Forest':       load_estimator('../models/tuned/random_forest_tuned.pkl'),
    'Gradient Boosting':   load_estimator('../models/tuned/gradient_boosting_tuned.pkl'),
    'Decision Tree':       load_estimator('../models/tuned/decision_tree_tuned.pkl'),
}

import json as _json
with open('../models/best_model_metadata.json') as _f:
    _meta = _json.load(_f)
print(f"Vitezny model (03d): {_meta['winning_model']}  ROC-AUC CV: {_meta['roc_auc_cv']:.4f}")
print(f"Modely: {list(tuned_models.keys())}")

import matplotlib as _mpl

# Paul Tol bright palette — colorblind-safe, perceptually uniform
_PALETTE = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377', '#BBBBBB']

_mpl.rcParams.update({
    # Color cycle
    'axes.prop_cycle':       _mpl.cycler(color=_PALETTE),

    # Backgrounds
    'figure.facecolor':      'white',
    'axes.facecolor':        '#F8F9FA',   # subtle off-white — "card" feel

    # Spines
    'axes.edgecolor':        '#DEE2E6',
    'axes.linewidth':        0.9,
    'axes.spines.top':       False,
    'axes.spines.right':     False,

    # Grid — solid, very light
    'axes.grid':             True,
    'axes.axisbelow':        True,
    'grid.color':            '#E9ECEF',
    'grid.linestyle':        '-',
    'grid.linewidth':        0.8,
    'grid.alpha':            1.0,

    # Typography
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'axes.titlesize':        14,
    'axes.titleweight':      'bold',
    'axes.titlepad':         14,
    'axes.labelsize':        12,
    'axes.labelweight':      'regular',
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'legend.fontsize':       10,
    'legend.title_fontsize': 11,
    'figure.titlesize':      16,
    'figure.titleweight':    'bold',

    # Text colors — near-black for contrast
    'text.color':            '#212529',
    'axes.labelcolor':       '#495057',
    'xtick.color':           '#495057',
    'ytick.color':           '#495057',

    # Ticks
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.minor.visible':   False,
    'ytick.minor.visible':   False,
    'xtick.major.pad':       5,
    'ytick.major.pad':       5,

    # Lines & markers
    'lines.linewidth':       2.0,
    'lines.markersize':      7,
    'patch.linewidth':       0.6,

    # Legend
    'legend.frameon':        True,
    'legend.framealpha':     0.92,
    'legend.edgecolor':      '#DEE2E6',
    'legend.fancybox':       True,
    'legend.borderpad':      0.6,

    # Figure / saving
    'figure.dpi':            100,
    'figure.figsize':        [8, 5],
    'savefig.dpi':           150,
    'savefig.bbox':          'tight',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'none',
})


Vitezny model (03d): Logistic Regression  ROC-AUC CV: 0.8000
Modely: ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'Decision Tree']


## Matice nákladů (Cost Matrix) — finanční dopad natrénovaných modelů
Accuracy ignoruje asymetrii chyb. Klíčová metrika je **Recall pro třídu Dropout** a **čistý finanční přínos**.

**Finanční hodnoty:**

| | Predikce: Dostuduje (0) | Predikce: Dropout (1) |
|---|---|---|
| **Skutečnost: Dostuduje (0)** | TN = 0 Kč | FP = −5 000 Kč |
| **Skutečnost: Dropout (1)** | FN = −50 000 Kč | TP = +45 000 Kč |

Čistý přínos = TP × 45 000 − FP × 5 000 − FN × 50 000


In [2]:
BENEFIT_TP = 45_000
COST_FP    =  5_000
COST_FN    = 50_000

cost_rows = []
for name, model in tuned_models.items():
    y_pred  = model.predict(X_test_prep)
    y_proba = model.predict_proba(X_test_prep)[:, 1]
    cm      = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    net_value = tp * BENEFIT_TP - fp * COST_FP - fn * COST_FN
    recall    = recall_score(y_test, y_pred)
    roc_auc   = roc_auc_score(y_test, y_proba)

    cost_rows.append({
        'Model': name, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'Recall': round(recall, 3),
        'ROC-AUC': round(roc_auc, 4),
        'Čistý přínos (Kč)': f'{net_value:+,.0f} Kč',
        '_net_val': net_value,
    })

df_cost = pd.DataFrame(cost_rows).sort_values('_net_val', ascending=False).reset_index(drop=True)
print("--- Tuned modely: Recall, ROC-AUC a finanční přínos ---")
print(df_cost.drop(columns=['_net_val']).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in df_cost['_net_val']]
bars   = axes[0].bar(df_cost['Model'], df_cost['_net_val'], color=colors, alpha=0.85, edgecolor='black')
for bar, val in zip(bars, df_cost['_net_val']):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + abs(max(df_cost['_net_val'])) * 0.02,
                 f'{val:+,.0f} Kč', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Čistý finanční přínos — tuned modely', fontsize=12, pad=12)
axes[0].set_ylabel('Čistý přínos (Kč)')
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

axes[1].bar(df_cost['Model'], df_cost['Recall'], color='steelblue', alpha=0.85, edgecolor='black')
for i, (m, r) in enumerate(zip(df_cost['Model'], df_cost['Recall'])):
    axes[1].text(i, r + 0.01, f'{r:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title('Recall pro třídu Dropout — tuned modely', fontsize=12, pad=12)
axes[1].set_ylabel('Recall')
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
import os; os.makedirs('../results', exist_ok=True)
plt.savefig('../results/cost_matrix_tuned.png', dpi=120, bbox_inches='tight')
plt.show()
print("Graf uložen: results/cost_matrix_tuned.png")


--- Tuned modely: Recall, ROC-AUC a finanční přínos ---
              Model  TP  FP  FN   TN  Recall  ROC-AUC Čistý přínos (Kč)
      Decision Tree 334 385 137 1144   0.709   0.7798     +6,255,000 Kč
Logistic Regression 331 368 140 1161   0.703   0.8012     +6,055,000 Kč
      Random Forest 278 246 193 1283   0.590   0.7979     +1,630,000 Kč
  Gradient Boosting 222 158 249 1371   0.471   0.8024     -3,250,000 Kč


Graf uložen: results/cost_matrix_tuned.png


## Threshold Analysis — optimální klasifikační práh
Snížením prahu pod 0,5 zachytíme více dropoutů (vyšší Recall / TP) za cenu více falešných poplachů (FP).
Asymetrie nákladů (FN = 50 000 Kč vs. FP = 5 000 Kč) jednoznačně preferuje nižší práh.
Optimum se pohybuje kolem **0,35**.


In [3]:
thresholds  = np.arange(0.10, 0.91, 0.05)
thresh_rows = []

for name, model in tuned_models.items():
    y_proba = model.predict_proba(X_test_prep)[:, 1]
    for thresh in thresholds:
        y_pred_t        = (y_proba >= thresh).astype(int)
        cm              = confusion_matrix(y_test, y_pred_t, labels=[0, 1])
        tn, fp, fn, tp  = cm.ravel()
        net_value       = tp * BENEFIT_TP - fp * COST_FP - fn * COST_FN
        recall          = tp / (tp + fn) if (tp + fn) > 0 else 0
        precision       = tp / (tp + fp) if (tp + fp) > 0 else 0
        thresh_rows.append({
            'Model': name, 'Práh': round(thresh, 2),
            'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
            'Recall': round(recall, 3), 'Precision': round(precision, 3),
            'Čistý přínos (Kč)': net_value
        })

df_thresh = pd.DataFrame(thresh_rows)

best_thresholds = df_thresh.loc[df_thresh.groupby('Model')['Čistý přínos (Kč)'].idxmax()]
print("--- Optimální práh (maximální čistý přínos) ---")
print(best_thresholds[['Model', 'Práh', 'Recall', 'Precision', 'TP', 'FP', 'FN', 'Čistý přínos (Kč)']].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
for name, grp in df_thresh.groupby('Model'):
    ax.plot(grp['Práh'], grp['Čistý přínos (Kč)'], marker='o', markersize=4, linewidth=2, label=name)

ax.axvline(0.50, color='gray',    linestyle='--', linewidth=1.2, label='Výchozí práh (0.5)')
ax.axvline(0.35, color='crimson', linestyle='--', linewidth=1.8, label='Doporučený práh (0.35)')
ax.axhline(0,    color='black',   linewidth=0.7)
ax.set_title(
    f'Čistý finanční přínos vs. klasifikační práh\n'
    f'(TP=+{BENEFIT_TP:,} Kč, FP=−{COST_FP:,} Kč, FN=−{COST_FN:,} Kč)',
    fontsize=13, fontweight='bold', pad=14
)
ax.set_xlabel('Klasifikační práh')
ax.set_ylabel('Čistý přínos (Kč)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f} Kč'))
ax.legend()
plt.tight_layout()
plt.savefig('../results/threshold_analysis_tuned.png', dpi=120, bbox_inches='tight')
plt.show()
print("Graf uložen: results/threshold_analysis_tuned.png")


--- Optimální práh (maximální čistý přínos) ---
              Model  Práh  Recall  Precision  TP   FP  FN  Čistý přínos (Kč)
      Decision Tree  0.10   0.970      0.283 457 1156  14           14085000
  Gradient Boosting  0.10   0.932      0.311 439  973  32           13290000
Logistic Regression  0.15   0.960      0.295 452 1081  19           13985000
      Random Forest  0.10   0.977      0.263 460 1287  11           13715000


Graf uložen: results/threshold_analysis_tuned.png
